In [ ]:
!pip install langchain
!pip install langchain-openai
!pip install langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.4/404.4 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.0.0
    Uninstalling tenacity-9.0.0:
      Successfully uninstalled tenacity-9.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/3

In [ ]:
!unzip 'fest_data.zip' -d 'fest_data'

Archive:  fest_data.zip
   creating: fest_data/강동선사문화축제/
  inflating: fest_data/강동선사문화축제/daum_blog.xlsx  
  inflating: fest_data/강동선사문화축제/naver_blog.xlsx  
  inflating: fest_data/강동선사문화축제/naver_cafe.xlsx  
  inflating: fest_data/강동선사문화축제/total.xlsx  
   creating: fest_data/부평풍물대축제/
  inflating: fest_data/부평풍물대축제/daum_blog.xlsx  
  inflating: fest_data/부평풍물대축제/naver_blog.xlsx  
  inflating: fest_data/부평풍물대축제/naver_cafe.xlsx  
  inflating: fest_data/부평풍물대축제/total.xlsx  
   creating: fest_data/원주 댄싱카니발/
  inflating: fest_data/원주 댄싱카니발/daum_blog.xlsx  
  inflating: fest_data/원주 댄싱카니발/naver_blog.xlsx  
  inflating: fest_data/원주 댄싱카니발/naver_cafe.xlsx  
  inflating: fest_data/원주 댄싱카니발/total.xlsx  
   creating: fest_data/정읍 구절초꽃축제/
  inflating: fest_data/정읍 구절초꽃축제/daum_blog.xlsx  
  inflating: fest_data/정읍 구절초꽃축제/naver_blog.xlsx  
  inflating: fest_data/정읍 구절초꽃축제/naver_cafe.xlsx  
  inflating: fest_data/정읍 구절초꽃축제/total.xlsx  
   creating: fest_data/청원생명축제/
  inflating: fest_data/청원생명축제/daum_bl

In [ ]:
from langchain_openai import ChatOpenAI
with open('openai_api_key.txt', 'r') as f:
    key = f.read()


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document


In [ ]:
from langchain_community.document_loaders import DataFrameLoader

In [ ]:
import pandas as pd
import time
import os
FOLDER_PATH = 'fest_data'

In [ ]:
LLM = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.2, api_key=key)


In [ ]:
sp_sum = SystemMessagePromptTemplate.from_template("너는 한국어로 문서를 요약하는 대단한 한국어 문서 요약가야.")
hm_sum_1 = HumanMessagePromptTemplate.from_template("다음은 한 축제에 대한 리뷰야. 이것을 한국어로 요약해줘. 거짓 정보는 빼 줘. 광고 같다면 광고라고 표기해 줘. 요약할 때 1문장 으로 정리해 줘: {context}")

prompt_sum_1 = ChatPromptTemplate.from_messages(messages=[sp_sum, hm_sum_1])
chain_sum_1 = create_stuff_documents_chain(llm=LLM, prompt=prompt_sum_1)

In [ ]:
pt_sum_all = """다음 내용은 {festival} 에 대해 요약된 리뷰들이야.
전부 확인해서 한 문장으로 {festival}에 대한 내용을 정리해 줘. 한국어로 요약해 줘
단, 축제 자체의 정보 보다는 각 리뷰어들이 보고 겪은 것들을 중심으로 정리해 줘.
광고가 많다면 광고가 많다고 작성해 줘.
요약한 이후에, 모든 리뷰들을 살펴보았을 때, 전체적인 리뷰들이 내리는 평가를 100점 만점으로 할 때, 점수 1개로 정리해 줘. 긍정적일 수록 높은 점수, 부정적일 수록 낮은 점수를 줘
네 답변의 형식은: 요약; 평가: 점수 꼴이야.
: {context}"""
hm_sum_all = HumanMessagePromptTemplate.from_template(pt_sum_all)

prompt_sum_all = ChatPromptTemplate.from_messages(messages=[sp_sum, hm_sum_all])
chain_sum_all = create_stuff_documents_chain(llm=LLM, prompt=prompt_sum_all)

In [ ]:
pt_last = """다음은 각 사이트별 요약된 리뷰들이야.
이를 요약해서 전체적인 {festival}에 대한 리뷰를 1문장으로 작성해 줘.
추가로, 각 요악 끝엔 점수가 있어. 이 점수들을 바탕으로 최종 점수를 부여해 줘.
100점 만점으로 할 때, 긍정적일 수록 높은 점수, 부정적일 수록 낮은 점수를 줘.
네 답변의 형식은: 요약; 평가: 점수 꼴이야.
: {context}"""
hm_last = HumanMessagePromptTemplate.from_template(pt_last)

prompt_last = ChatPromptTemplate.from_messages(messages=[sp_sum, hm_last])
chain_last = create_stuff_documents_chain(llm=LLM, prompt=prompt_last)

In [ ]:
SUMMARIES = pd.DataFrame(columns=['festival', 'summary', 's_naver_cafe', 's_naver_blog', 's_daum_blog', 'numbers_net', 'n_nc', 'n_nb', 'n_db'])
summaries = []

In [ ]:
for festival in os.listdir(FOLDER_PATH):
    d = {}
    d['festival'] = festival
    tt = os.path.join(FOLDER_PATH, festival)
    nb = pd.read_excel(os.path.join(tt, 'naver_blog.xlsx'))
    nc = pd.read_excel(os.path.join(tt, 'naver_cafe.xlsx'))
    db = pd.read_excel(os.path.join(tt, 'daum_blog.xlsx'))
    d['n_nb'] = len(nb)
    d['n_nc'] = len(nc)
    d['n_db'] = len(db)
    d['numbers_net'] = d['n_nb'] + d['n_nc'] + d['n_db']
    d['s_naver_cafe'] = chain_sum_all.invoke({'context': DataFrameLoader(nc, page_content_column='Content').load(), 'festival': festival})
    time.sleep(5)
    d['s_naver_blog'] = chain_sum_all.invoke({'context': DataFrameLoader(nb, page_content_column='Content').load(), 'festival': festival})
    time.sleep(5)
    d['s_daum_blog'] = chain_sum_all.invoke({'context': DataFrameLoader(db, page_content_column='Content').load(), 'festival': festival})
    time.sleep(5)
    text = f"naver_blog: {d['s_naver_blog']}\nnaver_cafe: {d['s_naver_cafe']}\ndaum_blog: {d['s_daum_blog']}"
    doc = Document(page_content=text)
    d['summary']=chain_last.invoke({'context': [doc], 'festival': festival})
    summaries.append(d)
    print(d)
    time.sleep(5)
SUMMARIES = pd.concat([SUMMARIES, pd.DataFrame(summaries)])

{'festival': '횡성한우축제', 'n_nb': 31, 'n_nc': 9, 'n_db': 85, 'numbers_net': 125, 's_naver_cafe': '횡성한우축제는 다양한 프로그램과 공연, 맛있는 음식이 가득한 축제로, 특히 한우 요리의 품질에 대한 아쉬움이 있었지만 가족 단위 방문객들에게는 즐거운 경험으로 평가되었으며, 광고가 많다는 의견도 있었다. 평가: 75점', 's_naver_blog': '횡성한우축제는 다양한 먹거리와 볼거리가 풍성하지만, 주차 공간이 협소하고 광고가 많아 혼잡한 분위기 속에서 한우를 저렴하게 즐길 수 있는 기회를 제공하는 축제이다; 평가: 75점', 's_daum_blog': '횡성한우축제는 2023년 10월 6일부터 10일까지 강원도 횡성종합운동장에서 열리며, 다양한 먹거리와 볼거리가 가득한 축제로, 특히 횡성한우를 저렴하게 즐길 수 있는 기회가 많고, 셔틀버스 운영으로 접근성이 좋지만, 행사장 혼잡과 광고가 많다는 의견도 있다; 평가: 85점', 'summary': '횡성한우축제는 다양한 먹거리와 볼거리가 풍성하며, 한우를 저렴하게 즐길 수 있는 기회를 제공하지만, 주차 공간 부족과 혼잡한 분위기, 광고가 많다는 아쉬움이 있는 축제이다; 평가: 78.33점'}
{'festival': '정읍 구절초꽃축제', 'n_nb': 12, 'n_nc': 3, 'n_db': 88, 'numbers_net': 103, 's_naver_cafe': '정읍 구절초꽃축제는 아름다운 꽃길과 자연환경 속에서 힐링을 경험할 수 있는 축제로, 많은 사람들이 방문하여 꽃을 감상하며 마음의 정화를 느끼는 기회를 제공하지만, 광고가 다소 많다는 의견도 있다; 평가: 85점.', 's_naver_blog': '정읍 구절초꽃축제는 다양한 가을꽃과 함께 구절초를 감상할 수 있는 행사로, 개막식과 공연이 열리며, 축제 기간 동안 한산한 분위기 속에서 방문객들이 즐길 수 있는 다양한 볼거리가 마련되어 있지만, 광고가 많고 구절초의 개화 상태가 기대에 

In [ ]:
summaries

[{'festival': '횡성한우축제',
  'n_nb': 31,
  'n_nc': 9,
  'n_db': 85,
  'numbers_net': 125,
  's_naver_cafe': '횡성한우축제는 다양한 프로그램과 공연, 맛있는 음식이 가득한 축제로, 특히 한우 요리의 품질에 대한 아쉬움이 있었지만 가족 단위 방문객들에게는 즐거운 경험으로 평가되었으며, 광고가 많다는 의견도 있었다. 평가: 75점',
  's_naver_blog': '횡성한우축제는 다양한 먹거리와 볼거리가 풍성하지만, 주차 공간이 협소하고 광고가 많아 혼잡한 분위기 속에서 한우를 저렴하게 즐길 수 있는 기회를 제공하는 축제이다; 평가: 75점',
  's_daum_blog': '횡성한우축제는 2023년 10월 6일부터 10일까지 강원도 횡성종합운동장에서 열리며, 다양한 먹거리와 볼거리가 가득한 축제로, 특히 횡성한우를 저렴하게 즐길 수 있는 기회가 많고, 셔틀버스 운영으로 접근성이 좋지만, 행사장 혼잡과 광고가 많다는 의견도 있다; 평가: 85점',
  'summary': '횡성한우축제는 다양한 먹거리와 볼거리가 풍성하며, 한우를 저렴하게 즐길 수 있는 기회를 제공하지만, 주차 공간 부족과 혼잡한 분위기, 광고가 많다는 아쉬움이 있는 축제이다; 평가: 78.33점'},
 {'festival': '정읍 구절초꽃축제',
  'n_nb': 12,
  'n_nc': 3,
  'n_db': 88,
  'numbers_net': 103,
  's_naver_cafe': '정읍 구절초꽃축제는 아름다운 꽃길과 자연환경 속에서 힐링을 경험할 수 있는 축제로, 많은 사람들이 방문하여 꽃을 감상하며 마음의 정화를 느끼는 기회를 제공하지만, 광고가 다소 많다는 의견도 있다; 평가: 85점.',
  's_naver_blog': '정읍 구절초꽃축제는 다양한 가을꽃과 함께 구절초를 감상할 수 있는 행사로, 개막식과 공연이 열리며, 축제 기간 동안 한산한 분위기 속에서 방문객들이 즐길 수 있는 다양한 볼거리가 마련

In [ ]:
summs = pd.DataFrame(summaries)
summs['score'] = summs['summary'].apply(lambda x: x.split('평가')[-1].split(':')[1].split('\n')[0])
summs['score_nc'] = summs['s_naver_cafe'].apply(lambda x: x.split('평가')[-1].split(':')[1].split('\n')[0])
summs['score_nb'] = summs['s_naver_blog'].apply(lambda x: x.split('평가')[-1].split(':')[1].split('\n')[0])
summs['score_db'] = summs['s_daum_blog'].apply(lambda x: x.split('평가')[-1].split(':')[1].split('\n')[0])
summs.sort_values(by='score', ascending=False)

,festival,n_nb,n_nc,n_db,numbers_net,s_naver_cafe,s_naver_blog,s_daum_blog,summary,score,score_nc,score_nb,score_db
3,청원생명축제,23,7,90,120,청원생명축제는 다양한 체험 부스와 먹거리가 풍부하여 가족과 함께 즐길 수 있는 알찬...,"청원생명축제는 가족 단위 방문객들에게 다양한 체험과 즐길 거리를 제공하며, 특히 포...","청원생명축제는 충북 청주시 청원구 오창읍 미래지농촌테마공원에서 열리는 축제로, 다양...",청원생명축제는 가족 단위 방문객들에게 다양한 체험과 풍부한 먹거리를 제공하며 가성비...,85점.,85점,85점.,85점
2,부평풍물대축제,20,2,88,110,부평풍물대축제는 다양한 공연과 체험 프로그램이 마련되어 있어 가족 단위 방문객들이 ...,부평풍물대축제는 다양한 체험과 공연이 마련되어 있어 가족 단위 방문객들에게 즐거움을...,부평풍물대축제는 2023년 9월 22일부터 24일까지 부평대로에서 열린 대규모 축제...,부평풍물대축제는 다양한 공연과 체험 프로그램으로 가족 단위 방문객들에게 즐거움을 주...,83.33점,80점,85점,85점
4,강동선사문화축제,13,3,85,101,강동선사문화축제는 가족 단위 방문객들에게 다양한 체험 프로그램과 맛있는 먹거리를 제...,"강동선사문화축제는 가족 단위 방문객들이 많고, 다양한 체험 프로그램과 먹거리가 마련...","강동선사문화축제는 매년 10월에 서울 암사동 유적지에서 열리며, 다양한 볼거리와 먹...",강동선사문화축제는 가족 단위 방문객들에게 다양한 체험 프로그램과 먹거리를 제공하며 ...,80점.,85점,75점,80점
5,원주 댄싱카니발,14,2,87,103,"원주 댄싱카니발은 12년째를 맞이하여 새로운 변화로 긍정적인 반응을 얻었지만, 축제...","원주 댄싱카니발은 다양한 댄스팀의 공연과 함께 프리마켓, 체험부스 등 다채로운 부대...",원주 댄싱카니발은 2023년 9월 22일부터 24일까지 원주 댄싱공연장 일대에서 열...,"원주 댄싱카니발은 다양한 공연과 부대행사가 마련되어 관람객들에게 즐거움을 주지만, ...",80점.,75점.,80점,85점
1,정읍 구절초꽃축제,12,3,88,103,"정읍 구절초꽃축제는 아름다운 꽃길과 자연환경 속에서 힐링을 경험할 수 있는 축제로,...","정읍 구절초꽃축제는 다양한 가을꽃과 함께 구절초를 감상할 수 있는 행사로, 개막식과...","정읍 구절초 꽃축제는 2023년 10월 5일부터 10월 15일까지 진행되었으며, 방...","정읍 구절초꽃축제는 아름다운 꽃과 다양한 체험 프로그램을 통해 힐링을 제공하지만, ...",78.33점.,85점.,70점,80점
0,횡성한우축제,31,9,85,125,"횡성한우축제는 다양한 프로그램과 공연, 맛있는 음식이 가득한 축제로, 특히 한우 요...","횡성한우축제는 다양한 먹거리와 볼거리가 풍성하지만, 주차 공간이 협소하고 광고가 많...",횡성한우축제는 2023년 10월 6일부터 10일까지 강원도 횡성종합운동장에서 열리며...,"횡성한우축제는 다양한 먹거리와 볼거리가 풍성하며, 한우를 저렴하게 즐길 수 있는 기...",78.33점,75점,75점,85점


In [ ]:
summs.to_excel('fest_summaries.xlsx')